# 09 — Descriptors and Properties

Descriptors are the mechanism behind `property`, `classmethod`,
`staticmethod`, `functools.cached_property`, `__slots__`, and every ORM
field you have ever used. They are also the cleanest way to factor out
repeated attribute logic. Once you see that a plain method is *also* a
descriptor, the whole object system clicks into place.

`#descriptors` `#properties` `#attributes` `#protocols`

<sub>Setup cell — run it once per session so `check()` works.</sub>

In [ ]:
# Wire the grader up. Run this cell first, every session.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents)
              if (p / "tutorial" / "grader.py").exists()), None)
if _root is None:
    raise RuntimeError("run this notebook from inside the tutorial repository")
sys.path.insert(0, str(_root))

from tutorial.grader import check, hint, solution   # noqa: E402

# A notebook kernel already owns an event loop, so asyncio.run() would refuse
# to start another one. Hand those coroutines a private loop on a worker
# thread instead, so the asyncio lessons run here exactly as they do in a
# script. (Outside a notebook this wrapper is a no-op.)
import asyncio as _asyncio                                    # noqa: E402
from concurrent.futures import ThreadPoolExecutor as _Pool    # noqa: E402

_plain_run = _asyncio.run


def _run_anywhere(coro, **kwargs):
    try:
        _asyncio.get_running_loop()
    except RuntimeError:
        return _plain_run(coro, **kwargs)
    with _Pool(1) as pool:
        return pool.submit(_plain_run, coro, **kwargs).result()


_asyncio.run = _run_anywhere

print("grader ready — Python", sys.version.split()[0])

## The protocol

A **descriptor** is any object whose *class* defines at least one of:

| method | called for |
|---|---|
| `__get__(self, obj, objtype)` | attribute read |
| `__set__(self, obj, value)` | attribute write |
| `__delete__(self, obj)` | `del obj.attr` |
| `__set_name__(self, owner, name)` | at class creation, to learn its own name |

- **data descriptor** — defines `__set__` and/or `__delete__`.
  Takes precedence over the instance `__dict__`.
- **non-data descriptor** — only `__get__`. The instance dict wins.

Descriptors only work when placed on a **class**, never on an instance.

In [ ]:
class Loud:
    def __set_name__(self, owner, name):
        print(f"  __set_name__: I live on {owner.__name__} as {name!r}")
        self.name = name
    def __get__(self, obj, objtype=None):
        print(f"  __get__(obj={obj!r}, objtype={objtype.__name__})")
        if obj is None:
            return self               # accessed on the class
        return obj.__dict__.get(self.name, "unset")
    def __set__(self, obj, value):
        print(f"  __set__({value!r})")
        obj.__dict__[self.name] = value

class Thing:
    field = Loud()

print("--- class creation happened above ---")
t = Thing()
t.field = 42
print("read:", t.field)
print("on the class:", Thing.field)

`__set_name__` (3.6+) is what lets a descriptor learn the attribute
name it was assigned to — before it existed, every ORM made you
write `name = Field("name")`.

When accessed on the class rather than an instance, `obj` is
`None`; returning `self` is the convention so that
`Thing.field` gives you the descriptor for introspection.

### ❓ Predict the output

Read the cell below and decide what it prints **before** running it. Then run it.

In [ ]:
class NonData:
    def __get__(self, obj, objtype=None): return "from descriptor"

class Data(NonData):
    def __set__(self, obj, value): obj.__dict__["d"] = value

class C:
    n = NonData()
    d = Data()

c = C()
c.__dict__["n"] = "from instance dict"
c.__dict__["d"] = "from instance dict"
print(c.n)
print(c.d)

<details>
<summary><b>the answer, and why</b></summary>

```
from instance dict
from descriptor
```

The non-data descriptor loses to the instance dict; the data
descriptor wins. This single precedence rule is why `property`
can enforce validation that an instance attribute cannot bypass,
and why methods (non-data) can be shadowed per instance.

</details>

## Functions are descriptors — that is how `self` works

A plain function stored on a class is a non-data descriptor. Accessing
it through an instance calls `function.__get__(instance, cls)`, which
returns a **bound method** — a small object holding the function and
the instance. Calling it inserts the instance as the first argument.

In [ ]:
class C:
    def method(self, x): return f"method({self!r}, {x})"

c = C()
print("via instance:", type(c.method).__name__)
print("via class   :", type(C.method).__name__)
print("bound to    :", c.method.__self__ is c)
print("underlying  :", c.method.__func__ is C.method)

# Binding by hand -- exactly what attribute access does:
manual = C.method.__get__(c, C)
print("manual bind:", manual(1))

# classmethod and staticmethod are descriptors too -- but
# different ones. A plain method is stored as a bare function:
print("in the class dict:", type(C.__dict__["method"]).__name__)

class D:
    @staticmethod
    def s(): return "static"
    @classmethod
    def c(cls): return f"class {cls.__name__}"
print(type(D.__dict__["s"]).__name__, "->", D.s())
print(type(D.__dict__["c"]).__name__, "->", D.c())

Through an instance you get a `method`; through the class, the
plain `function`. A bound method is nothing more than that pair —
`__self__` holding the instance, `__func__` the original function
— and calling it slots `__self__` in as the first argument. The
manual `__get__` line is not an analogy for what attribute access
does; it is the same call.

The class `__dict__` shows the contrast. A plain method is stored
as a bare function, while `staticmethod` and `classmethod` are
**different descriptor objects** wrapping one. All three work
because each implements `__get__` its own way: the function binds
the instance, `classmethod` binds the class, and `staticmethod`
binds nothing and hands back the function untouched.

### 🧠 Quiz

Why does `obj.method` automatically receive `obj` as `self`?

- **a)** The interpreter special-cases attribute access on classes
- **b)** `function.__get__` returns a bound method that stores the instance
- **c)** `self` is a reserved keyword filled in by the compiler
- **d)** Methods are stored per instance, each capturing its own object

<details>
<summary><b>answer</b></summary>

**b** — Functions implement the descriptor protocol. `obj.method` is `type(obj).method.__get__(obj, type(obj))`, which produces the bound method.

</details>

## `property`: the descriptor you already use

`property` is a data descriptor implemented in C. Its constructor is
`property(fget, fset, fdel, doc)`; the `@x.setter` syntax just builds
a *new* property with one more function filled in.

In [ ]:
class Celsius:
    def __init__(self, degrees=0.0):
        self.degrees = degrees            # goes through the setter

    @property
    def degrees(self):
        return self._degrees

    @degrees.setter
    def degrees(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._degrees = float(value)

    @property
    def fahrenheit(self):                  # computed, read-only
        return self._degrees * 9 / 5 + 32

t = Celsius(25)
print(t.degrees, "C =", t.fahrenheit, "F")
t.degrees = 100
print("updated:", t.fahrenheit)

try:
    t.degrees = -300
except ValueError as exc:
    print("validated:", exc)
try:
    t.fahrenheit = 0
except AttributeError as exc:
    print("read-only:", exc)

# The equivalent without syntax sugar:
print("property is just an object:", type(Celsius.degrees).__name__,
      "| fset set:", Celsius.degrees.fset is not None)

Note that `__init__` assigns to `self.degrees`, not
`self._degrees` — so construction runs through the setter and
inherits the same validation as every later write. One place
enforces the invariant, and there is no way to build an invalid
instance.

`fahrenheit` is recomputed on every read and stored nowhere, so it
cannot drift out of step with `degrees`; having no setter, it
rejects assignment outright.

The last line is the point of the section: `property` is an
ordinary object with `fget`/`fset`/`fdel` attributes. `@x.setter`
mutates nothing — it constructs a **new** property with one more
function filled in and rebinds the name.

## When a property is not enough

Properties do not compose: validating ten fields the same way means
ten near-identical property pairs. A descriptor class is written once
and reused everywhere — that is the entire argument for them.

In [ ]:
class Typed:
    '''Reusable type-checked attribute.'''
    def __init__(self, expected_type, default=None):
        self.expected_type = expected_type
        self.default = default

    def __set_name__(self, owner, name):
        self.private = "_" + name
        self.public = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.private, self.default)

    def __set__(self, obj, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.public} must be {self.expected_type.__name__}, "
                f"got {type(value).__name__}"
            )
        setattr(obj, self.private, value)

class User:
    name = Typed(str, "")
    age = Typed(int, 0)
    tags = Typed(list)

    def __init__(self, name, age):
        self.name, self.age = name, age

u = User("ada", 36)
print(u.name, u.age, "| default:", u.tags)
try:
    u.age = "thirty"
except TypeError as exc:
    print("rejected:", exc)

Note the storage strategy: writing to `obj._age` rather than a dict
keyed by instance. Never store per-instance state on the
*descriptor itself* — there is exactly one descriptor object shared
by every instance of the class.

In [ ]:
# The classic bug: state on the descriptor instead of the instance.
class Broken:
    def __init__(self): self.value = None
    def __get__(self, obj, objtype=None): return self.value
    def __set__(self, obj, value): self.value = value   # SHARED!

class Item:
    field = Broken()

a, b = Item(), Item()
a.field = "a's value"
b.field = "b's value"
print("a.field =", a.field, "  <- clobbered by b")
print("same descriptor object:", Item.__dict__["field"] is Item.__dict__["field"])

There is exactly **one** descriptor object per class attribute,
created when the class body runs and shared by every instance ever
made. So `self.value` inside `__set__` is class-wide storage:
`a` and `b` wrote to the same box, and the second write won.

The failure is quiet, which is what makes it dangerous — nothing
raises, values simply bleed between unrelated objects, and it
usually surfaces first under concurrency or in a long-lived
process. Every correct descriptor keys its storage **by the
instance**: the instance `__dict__`, a private attribute name, or
a `WeakKeyDictionary`.

## Where to store the value

1. **Instance `__dict__` under the public name** — only safe for a
   *non-data* descriptor, or you get infinite recursion.
2. **Instance `__dict__` under a private name** (`_x`) — simple and
   debuggable. What most code does.
3. **A `WeakKeyDictionary` keyed by instance** — works with
   `__slots__`, requires hashable instances, adds indirection.

In [ ]:
import weakref

class WeakStored:
    def __set_name__(self, owner, name): self.name = name
    def __init__(self): self.data = weakref.WeakKeyDictionary()
    def __get__(self, obj, objtype=None):
        if obj is None: return self
        return self.data.get(obj)
    def __set__(self, obj, value): self.data[obj] = value

class Slotted:
    __slots__ = ("__weakref__",)        # needed for weak references
    x = WeakStored()

s = Slotted()
s.x = 42
print("works with __slots__:", s.x)
print("entries:", len(Slotted.x.data))
del s
print("after the instance dies:", len(Slotted.x.data))

The weak dictionary drops its entry automatically when the instance
is collected — a plain dict here would leak every object forever.

In [ ]:
# __delete__ and the full lifecycle.
class Tracked:
    def __set_name__(self, owner, name): self.private = "_" + name
    def __get__(self, obj, objtype=None):
        if obj is None: return self
        try:
            return getattr(obj, self.private)
        except AttributeError:
            raise AttributeError("value has been deleted") from None
    def __set__(self, obj, value): setattr(obj, self.private, value)
    def __delete__(self, obj):
        print("  __delete__ called")
        try:
            delattr(obj, self.private)
        except AttributeError:
            pass

class C:
    v = Tracked()

c = C()
c.v = 1
print("set:", c.v)
del c.v
try:
    c.v
except AttributeError as exc:
    print("after delete:", exc)

`__delete__` completes the protocol: on a data descriptor,
`del obj.attr` routes here instead of removing anything from the
instance dict. Doing the work properly means the attribute returns
to a genuinely *unset* state rather than lingering with a stale
value.

Two small choices carry weight. Raising `AttributeError` from
`__get__` is what makes `hasattr()` report `False` — any other
exception type propagates and the caller gets a crash instead of
an answer. And `from None` suppresses the chained internal error,
so the reader sees "value has been deleted" rather than a
"during handling of the above exception" sandwich.

---

## ⚒ Exercise — A configurable validating descriptor  
`09-descriptors/validated-field` · difficulty: **tricky**

Write `Field(*, type=None, min=None, max=None, choices=None,
default=MISSING, required=True)` — one descriptor covering the
validation you actually need.

- `type` → `TypeError` on mismatch (allow `int` where `float` is
  expected, but reject `bool` for both)
- `min` / `max` → `ValueError` when out of range (inclusive)
- `choices` → `ValueError` when not a member
- reading before any assignment returns `default`, or raises
  `AttributeError` when the field is `required` and has no default
- error messages must contain the attribute's name
- each instance keeps its own value; the descriptor holds none
- accessing the field on the **class** returns the descriptor
- `del obj.field` resets it to the unset state
- validation must run in this order: type, then choices, then range

Edit the next cell, run it, then run the check cell below it.

In [ ]:
MISSING = object()


class Field:
    def __init__(self, *, type=None, min=None, max=None,
                 choices=None, default=MISSING, required=True):
        ...

In [ ]:
check("09-descriptors/validated-field")

<details>
<summary><b>hints</b></summary>

1. `__set_name__` is where you learn the public name; derive a private one like `f'_field_{name}'` from it.
2. `isinstance(True, int)` is True — special-case `bool` explicitly.
3. For `type=float`, accept `int` as well: `expected = (int, float) if self.type is float else self.type`.
4. Reading an unset required field should raise AttributeError, so `hasattr()` correctly reports False for it.
5. Store into `obj.__dict__[self.private]` directly to avoid re-entering `__set__`.

</details>

<details>
<summary><b>reference solution</b></summary>

```python
MISSING = object()

class Field:
    def __init__(self, *, type=None, min=None, max=None,
                 choices=None, default=MISSING, required=True):
        self.type = type
        self.min = min
        self.max = max
        self.choices = choices
        self.default = default
        self.required = required

    def __set_name__(self, owner, name):
        self.name = name
        self.private = f"_field_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        if self.private in obj.__dict__:
            return obj.__dict__[self.private]
        if self.default is not MISSING:
            return self.default
        if self.required:
            raise AttributeError(f"{self.name!r} has no value")
        return None

    def __set__(self, obj, value):
        self.validate(value)
        obj.__dict__[self.private] = value

    def __delete__(self, obj):
        obj.__dict__.pop(self.private, None)

    def validate(self, value):
        if self.type is not None:
            expected = (int, float) if self.type is float else self.type
            if isinstance(value, bool) and self.type in (int, float):
                raise TypeError(
                    f"{self.name} must be {self.type.__name__}, got bool"
                )
            if not isinstance(value, expected):
                raise TypeError(
                    f"{self.name} must be {self.type.__name__}, "
                    f"got {builtin_type(value).__name__}"
                )
        if self.choices is not None and value not in self.choices:
            raise ValueError(
                f"{self.name} must be one of {list(self.choices)}, got {value!r}"
            )
        if self.min is not None and value < self.min:
            raise ValueError(f"{self.name} must be >= {self.min}, got {value!r}")
        if self.max is not None and value > self.max:
            raise ValueError(f"{self.name} must be <= {self.max}, got {value!r}")

builtin_type = type      # 'type' is shadowed by the parameter above
```

</details>

## Takeaways

- A descriptor is a class attribute implementing `__get__`/`__set__`/
  `__delete__`; `__set_name__` tells it its own name.
- Data descriptors beat the instance dict; non-data descriptors lose
  to it.
- Functions are non-data descriptors — that is the origin of `self`.
- Never keep per-instance state on the descriptor object.
- Use a property for one attribute, a descriptor class when the same
  logic repeats.

---

### ✅ Lesson complete

Exercises in this lesson: **1**. Check your standing across the whole course with:

```
uv run learn.py list
```